# Task 2.5 — Performance across user and item groups

Builds the report table for all models that have exported test lists:
baselines (Pop, Random), individual recommenders, and the three hybrids
(Hybrid Regression, Mixed Hybrid, Feature Augmentation).

Groups (no retraining): activity and mainstreaminess tertiles; PopLift for
niche users; miscalibration for low-activity users; middle-item coverage
(next 30% of the catalogue after the top 20% by training popularity).

## Prerequisites

```bash
python run_model.py Pop Random   # if missing
python run_mixed_hybrid.py
python run_hybrid_regression.py
python run_hybrid_feature_aug.py
python run_group_analysis.py --stage test
```

`run_group_analysis.py` writes (gitignored):

- `outputs/evaluation/test_user_group_analysis.csv`
- `outputs/evaluation/test_item_group_analysis.csv`

This notebook reads those CSVs and writes a summary under
`results/`.


## 0. Setup


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

EVAL = Path("outputs/evaluation")
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

USER_CSV = EVAL / "test_user_group_analysis.csv"
ITEM_CSV = EVAL / "test_item_group_analysis.csv"

for path in (USER_CSV, ITEM_CSV):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run: python run_group_analysis.py --stage test"
        )

users = pd.read_csv(USER_CSV)
items = pd.read_csv(ITEM_CSV)
print("models:", sorted(users["model"].unique()))
print("user rows:", len(users), "| item rows:", len(items))


models: ['BPR', 'ContentBased', 'EASE', 'HybridFeatureAug', 'HybridRegression', 'ItemKNN', 'LightGCN', 'MixedHybrid', 'Pop', 'Random', 'RecVAE', 'SLIMElastic', 'SimpleX', 'UserKNN']
user rows: 462 | item rows: 308


## 1. Report table


In [2]:
# Display order and labels used in the report
REPORT_MODELS = [
    ("Random", "Random"),
    ("Pop", "Pop"),
    ("UserKNN", "UserKNN"),
    ("ItemKNN", "ItemKNN"),
    ("EASE", "EASE"),
    ("SLIMElastic", "SLIMElastic"),
    ("BPR", "BPR"),
    ("LightGCN", "LightGCN"),
    ("SimpleX", "SimpleX"),
    ("RecVAE", "RecVAE"),
    ("ContentBased", "ContentBased"),
    ("HybridRegression", "Hybrid Regression"),
    ("MixedHybrid", "Mixed Hybrid"),
    ("HybridFeatureAug", "Feature Augmentation"),
]


def user_metric(model, grouping, group, metric):
    row = users[
        (users["model"] == model)
        & (users["grouping"] == grouping)
        & (users["group"] == group)
    ]
    if row.empty:
        return float("nan")
    return float(row.iloc[0][metric])


def middle_coverage(model):
    row = items[
        (items["model"] == model)
        & (items["grouping"] == "popularity")
        & (items["group"] == "middle")
    ]
    if row.empty:
        return float("nan")
    return float(row.iloc[0]["coverage@10"])


rows = []
missing = []
available = set(users["model"])
for model, label in REPORT_MODELS:
    if model not in available:
        missing.append(model)
        continue
    rows.append(
        {
            "model": label,
            "folder": model,
            "NDCG_low": user_metric(model, "activity", "low", "NDCG@10"),
            "NDCG_medium": user_metric(model, "activity", "medium", "NDCG@10"),
            "NDCG_high": user_metric(model, "activity", "high", "NDCG@10"),
            "NDCG_niche": user_metric(model, "mainstreaminess", "niche", "NDCG@10"),
            "NDCG_mainstream": user_metric(
                model, "mainstreaminess", "mainstream", "NDCG@10"
            ),
            "PopLift_niche": user_metric(
                model, "mainstreaminess", "niche", "PopLift@10"
            ),
            "Miscal_low_activity": user_metric(
                model, "activity", "low", "Miscalibration@10"
            ),
            "middle_coverage": middle_coverage(model),
        }
    )

table = pd.DataFrame(rows).set_index("model")
if missing:
    print("Missing (no recs_test / not in CSV):", missing)

display(
    table.drop(columns=["folder"]).style.format(
        {
            "NDCG_low": "{:.3f}",
            "NDCG_medium": "{:.3f}",
            "NDCG_high": "{:.3f}",
            "NDCG_niche": "{:.3f}",
            "NDCG_mainstream": "{:.3f}",
            "PopLift_niche": "{:.3f}",
            "Miscal_low_activity": "{:.3f}",
            "middle_coverage": "{:.0%}",
        }
    )
)


,NDCG_low,NDCG_medium,NDCG_high,NDCG_niche,NDCG_mainstream,PopLift_niche,Miscal_low_activity,middle_coverage
model,,,,,,,,
Random,0.004,0.004,0.019,0.014,0.004,-0.584,1.719,3%
Pop,0.087,0.101,0.178,0.133,0.115,2.009,1.238,0%
UserKNN,0.195,0.214,0.310,0.291,0.209,1.132,1.163,3%
ItemKNN,0.179,0.203,0.334,0.287,0.202,0.798,1.129,15%
EASE,0.221,0.243,0.380,0.343,0.233,0.968,1.058,7%
SLIMElastic,0.223,0.244,0.381,0.344,0.235,0.860,1.048,16%
BPR,0.206,0.207,0.360,0.314,0.209,0.666,1.168,28%
LightGCN,0.206,0.212,0.353,0.300,0.214,0.416,1.129,50%
SimpleX,0.208,0.221,0.335,0.293,0.212,0.825,1.027,29%


## 2. Group sizes (why activity gaps are not causal)


In [3]:
# Same for every model: groups are defined from the shared training split
sizes = (
    users[users["model"] == users["model"].iloc[0]]
    .loc[users["grouping"].isin(["activity", "mainstreaminess"])]
    [["grouping", "group", "n_users", "avg_train_items", "avg_test_items"]]
    .drop_duplicates()
    .sort_values(["grouping", "group"])
)
display(sizes)

act = sizes[sizes["grouping"] == "activity"].set_index("group")
ratio = act.loc["high", "avg_test_items"] / act.loc["low", "avg_test_items"]
print(f"high / low avg test items ≈ {ratio:.1f}×")


,grouping,group,n_users,avg_train_items,avg_test_items
0,activity,high,315,145.688889,17.653968
1,activity,low,314,19.031847,1.847134
2,activity,medium,314,47.770701,5.366242
3,mainstreaminess,average,314,65.636943,7.617834
4,mainstreaminess,mainstream,315,32.993651,3.571429
5,mainstreaminess,niche,314,114.219745,13.722930


high / low avg test items ≈ 9.6×


## 3. Save tracked artifacts

Writes under `results/` (not gitignored).


In [4]:
out_csv = RESULTS / "task2_5_group_table.csv"
table.drop(columns=["folder"]).to_csv(out_csv, float_format="%.6f")
print("wrote", out_csv)

# LaTeX rows matching RecSys_Final_Project/sections/task2-5.tex
def latex_num(x):
    if pd.isna(x):
        return "--"
    if x < 0:
        return f"$-${abs(x):.3f}"
    return f"{x:.3f}"


lines = []
for model, label in REPORT_MODELS:
    if model not in available:
        continue
    r = table.loc[label]
    mid = "--" if pd.isna(r["middle_coverage"]) else f"{100 * r['middle_coverage']:.0f}\\%"
    lines.append(
        f"{label:20} & {latex_num(r['NDCG_low'])} & {latex_num(r['NDCG_medium'])} & "
        f"{latex_num(r['NDCG_high'])} & {latex_num(r['NDCG_niche'])} & "
        f"{latex_num(r['NDCG_mainstream'])} & {latex_num(r['PopLift_niche'])} & "
        f"{latex_num(r['Miscal_low_activity'])} & {mid} \\\\"
    )

out_tex = RESULTS / "task2_5_group_table_rows.tex"
out_tex.write_text("\n".join(lines) + "\n")
print("wrote", out_tex)
print("\n".join(lines))


wrote results\task2_5_group_table.csv
wrote results\task2_5_group_table_rows.tex
Random               & 0.004 & 0.004 & 0.019 & 0.014 & 0.004 & $-$0.584 & 1.719 & 3\% \\
Pop                  & 0.087 & 0.101 & 0.178 & 0.133 & 0.115 & 2.009 & 1.238 & 0\% \\
UserKNN              & 0.195 & 0.214 & 0.310 & 0.291 & 0.209 & 1.132 & 1.163 & 3\% \\
ItemKNN              & 0.179 & 0.203 & 0.334 & 0.287 & 0.202 & 0.798 & 1.129 & 15\% \\
EASE                 & 0.221 & 0.243 & 0.380 & 0.343 & 0.233 & 0.968 & 1.058 & 7\% \\
SLIMElastic          & 0.223 & 0.244 & 0.381 & 0.344 & 0.235 & 0.860 & 1.048 & 16\% \\
BPR                  & 0.206 & 0.207 & 0.360 & 0.314 & 0.209 & 0.666 & 1.168 & 28\% \\
LightGCN             & 0.206 & 0.212 & 0.353 & 0.300 & 0.214 & 0.416 & 1.129 & 50\% \\
SimpleX              & 0.208 & 0.221 & 0.335 & 0.293 & 0.212 & 0.825 & 1.027 & 29\% \\
RecVAE               & 0.211 & 0.228 & 0.370 & 0.319 & 0.225 & 0.584 & 1.061 & 40\% \\
ContentBased         & 0.028 & 0.034 & 0.073 & 0.0

## 4. Optional: re-run group analysis from this notebook

Uncomment to regenerate the CSVs for every model that has `recs_test.tsv`.


In [5]:
# !python run_group_analysis.py --stage test
